# ParFlow-CLM vs Ameriflux Comparison

Load model output from Notebook 2, fetch Ameriflux tower observations, and evaluate:
- Daily latent heat time series
- Scatter plot with 1:1 line
- ET component breakdown
- Soil moisture
- Forcing quality check (CW3E vs Ameriflux downward shortwave)
- Metrics: KGE, NSE, RMSE, bias

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import hf_hydrodata as hf
from pathlib import Path
from datetime import datetime

from helpers import read_site_config

# HydroData PIN (persists from Notebook 1, but register again if needed)
hf.register_api_pin("your_email@example.com", "your_pin")

In [ ]:
# ============================================================
# Load Site Configuration
# ============================================================
static_write_dir = "/Users/reed/Projects/single_column_PFCLM/runs/US_Slt_WY2012"

run_dir = Path(static_write_dir)
cfg = read_site_config(run_dir / "site_config.txt")

site_id    = cfg["site_id"]
water_year = int(cfg["water_year"])
start      = cfg["start"]
end        = cfg["end"]
lat        = float(cfg["lat"])
lon        = float(cfg["lon"])
utc_offset = int(cfg["utc_offset"])
porosity   = float(cfg["soil_porosity"])

print(f"Site: {site_id}, WY{water_year}, UTC offset: {utc_offset}")

In [ ]:
# ============================================================
# Load Model Output
# ============================================================
run_dir = Path(static_write_dir)
clm_files = sorted(run_dir.glob("*.out.CLM.*.nc"))
pf_files = sorted(run_dir.glob("*.out.00001.nc"))

if not clm_files:
    raise FileNotFoundError(f"No CLM output in {run_dir}. Run Notebook 2 first.")

clm_ds = xr.open_dataset(clm_files[0])
n_steps = len(clm_ds["eflx_lh_tot"].values.flatten())

# Build datetime index with CLM end-of-hour shift
# CLM timestamps are end-of-hour: value at T represents avg during T-1 to T
# Shift back 1h so timestamps align with the hour the flux occurred in
dt_start = datetime.strptime(start, "%Y-%m-%d")
model_dates = pd.date_range(dt_start, periods=n_steps, freq="h") - pd.Timedelta(hours=1)

# Extract hourly LH
model_lh_hourly = pd.Series(
    clm_ds["eflx_lh_tot"].values.flatten(),
    index=model_dates, name="model_lh"
)

print(f"Loaded {n_steps} hourly time steps")
print(f"  LH range: {model_lh_hourly.min():.1f} to {model_lh_hourly.max():.1f} W/m2")

In [ ]:
# ============================================================
# Fetch Ameriflux Observations
# ============================================================
obs_df = hf.get_point_data(
    dataset="ameriflux", variable="latent_heat",
    temporal_resolution="hourly", aggregation="sum",
    date_start=start, date_end=end, site_ids=site_id,
)

obs_dates = pd.to_datetime(obs_df["date"])
obs_lh_hourly = pd.Series(
    obs_df[site_id].values.astype(float),
    index=pd.DatetimeIndex(obs_dates), name="obs_lh"
)
obs_lh_hourly = obs_lh_hourly.replace(-9999, np.nan).dropna()

print(f"Ameriflux observations: {len(obs_lh_hourly)} valid hourly values")
print(f"  LH range: {obs_lh_hourly.min():.1f} to {obs_lh_hourly.max():.1f} W/m2")

In [ ]:
# ============================================================
# Resample to Daily Means (in local time)
# ============================================================
# Shift to local time for daily binning (midnight-to-midnight local),
# then shift index back so dates stay in UTC calendar.

def hourly_to_daily(hourly, utc_off):
    """Resample hourly UTC series to daily means in local time."""
    shifted = hourly.copy()
    delta = pd.Timedelta(hours=utc_off)
    shifted.index = shifted.index + delta
    daily = shifted.resample("D").mean()
    daily.index = daily.index - delta
    return daily.dropna()

obs_daily = hourly_to_daily(obs_lh_hourly, utc_offset)
mod_daily = hourly_to_daily(model_lh_hourly, utc_offset)

print(f"Daily obs: {len(obs_daily)} days")
print(f"Daily mod: {len(mod_daily)} days")

In [ ]:
# ============================================================
# Compute Metrics
# ============================================================
common = obs_daily.index.intersection(mod_daily.index)
obs_c = obs_daily.loc[common].dropna()
mod_c = mod_daily.loc[common].dropna()
common = obs_c.index.intersection(mod_c.index)
obs_c = obs_c.loc[common]
mod_c = mod_c.loc[common]

bias = float((mod_c - obs_c).mean())
rmse = float(np.sqrt(((mod_c - obs_c) ** 2).mean()))
r = float(np.corrcoef(obs_c, mod_c)[0, 1])

# KGE (Gupta et al. 2009)
mu_o, mu_m = obs_c.mean(), mod_c.mean()
sigma_o, sigma_m = obs_c.std(), mod_c.std()
kge = float(1 - np.sqrt((r - 1)**2 + (sigma_m/sigma_o - 1)**2 + (mu_m/mu_o - 1)**2))

# NSE
ss_res = ((mod_c - obs_c) ** 2).sum()
ss_tot = ((obs_c - mu_o) ** 2).sum()
nse = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

# Annual ET (from daily LH)
lv = 2.45e6  # J/kg (latent heat of vaporization)
obs_et = float((obs_c / lv * 86400).sum())  # mm/yr
mod_et = float((mod_c / lv * 86400).sum())

print(f"\\n{'='*50}")
print(f"  {site_id} WY{water_year} — Daily LH Metrics")
print(f"{'='*50}")
print(f"  N days:     {len(common)}")
print(f"  Bias:       {bias:+.1f} W/m2")
print(f"  RMSE:       {rmse:.1f} W/m2")
print(f"  r:          {r:.3f}")
print(f"  KGE:        {kge:.3f}")
print(f"  NSE:        {nse:.3f}")
print(f"  Obs ET:     {obs_et:.0f} mm/yr")
print(f"  Model ET:   {mod_et:.0f} mm/yr")
print(f"{'='*50}")

In [ ]:
# ============================================================
# Time Series Plot — Daily Latent Heat
# ============================================================
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(obs_c.index, obs_c.values, color="black", lw=1.0, label="Ameriflux Obs", alpha=0.8)
ax.plot(mod_c.index, mod_c.values, color="steelblue", lw=1.0, label="PF-CLM Model", alpha=0.8)

ax.set_ylabel("Latent Heat [W/m2]")
ax.set_title(f"{site_id} WY{water_year} — Daily Mean LH  |  "
             f"KGE={kge:.2f}, Bias={bias:+.1f} W/m2, RMSE={rmse:.1f}")
ax.legend(loc="upper right")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.axhline(0, color="gray", lw=0.5, ls="--")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Scatter Plot — Observed vs Modeled
# ============================================================
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(obs_c.values, mod_c.values, s=10, alpha=0.5, color="steelblue", edgecolors="none")

# 1:1 line
lim_max = max(obs_c.max(), mod_c.max()) * 1.1
ax.plot([0, lim_max], [0, lim_max], "k--", lw=1, label="1:1")
ax.set_xlim(0, lim_max)
ax.set_ylim(0, lim_max)

ax.set_xlabel("Ameriflux LH [W/m2]")
ax.set_ylabel("PF-CLM LH [W/m2]")
ax.set_title(f"{site_id} WY{water_year}  |  r={r:.2f}, KGE={kge:.2f}")
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ET Components — Stacked Area + Obs Overlay
# ============================================================
et_vars = {
    "qflx_tran_veg": ("Transpiration", "green"),
    "qflx_evap_soi": ("Soil Evaporation", "sienna"),
    "qflx_evap_veg": ("Canopy Evaporation", "dodgerblue"),
}

fig, ax = plt.subplots(figsize=(14, 5))

# Convert each component to daily LH (W/m2) using Lv
lv = 2.45e6
et_daily = {}
for var, (label, color) in et_vars.items():
    if var in clm_ds:
        hourly = pd.Series(clm_ds[var].values.flatten(), index=model_dates)
        daily = hourly_to_daily(hourly, utc_offset) * lv  # mm/s -> W/m2
        et_daily[label] = daily

if et_daily:
    et_df = pd.DataFrame(et_daily).reindex(mod_daily.index).fillna(0)
    colors = [et_vars[v][1] for v in et_vars if et_vars[v][0] in et_df.columns]
    ax.stackplot(et_df.index, *[et_df[col] for col in et_df.columns],
                 labels=et_df.columns, colors=colors, alpha=0.7)

ax.plot(obs_c.index, obs_c.values, "k-", lw=1.0, label="Ameriflux Obs")
ax.set_ylabel("Latent Heat [W/m2]")
ax.set_title(f"{site_id} WY{water_year} — ET Components")
ax.legend(loc="upper right", fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Soil Moisture — Top 2 Layers from ParFlow
# ============================================================
if pf_files:
    pf_ds = xr.open_dataset(pf_files[0])
    if "saturation" in pf_ds:
        sat = pf_ds["saturation"]
        pf_dates = pd.date_range(start, periods=sat.shape[0], freq="h")

        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(pf_dates, sat[:, 9, 0, 0].values * porosity * 100,
                lw=0.5, color="steelblue", label="5 cm (cell center)")
        ax.plot(pf_dates, sat[:, 8, 0, 0].values * porosity * 100,
                lw=0.5, color="navy", label="25 cm (cell center)")
        ax.set_ylabel("Vol. Water Content [%]")
        ax.set_title(f"{site_id} WY{water_year} — Soil Moisture (porosity={porosity})")
        ax.legend()
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
        plt.tight_layout()
        plt.show()
    pf_ds.close()
else:
    print("No ParFlow pressure/saturation NetCDF output found.")

In [ ]:
# ============================================================
# Forcing Quality — CW3E vs Ameriflux (5 variables)
# ============================================================
# Compare: SWdown, LWdown, Tair, Wind speed, VPD
# Not all variables are available at all sites — skips missing ones.

# Load CW3E forcing
forcing_path = run_dir / cfg["forcing_file"]
forcing_raw = np.loadtxt(forcing_path)
forc_dates = pd.date_range(start, periods=len(forcing_raw), freq="h")
# Shift to local time for diurnal/daily comparisons
forc_dates_local = forc_dates + pd.Timedelta(hours=utc_offset)

forcing_df = pd.DataFrame(forcing_raw, index=forc_dates_local,
    columns=["dswr", "dlwr", "precip", "temp_k", "ugrd", "vgrd", "press", "spfh"])
forcing_df["temp_c"] = forcing_df["temp_k"] - 273.15
forcing_df["wind_speed"] = np.sqrt(forcing_df["ugrd"]**2 + forcing_df["vgrd"]**2)

# VPD: e = q*P/(0.622+0.378*q), esat = 611.2*exp(17.67*Tc/(Tc+243.5)), VPD = (esat-e)/100
q, P, Tc = forcing_df["spfh"], forcing_df["press"], forcing_df["temp_c"]
e = q * P / (0.622 + 0.378 * q)
esat = 611.2 * np.exp(17.67 * Tc / (Tc + 243.5))
forcing_df["vpd_hpa"] = ((esat - e) / 100.0).clip(lower=0)

# Variables to compare: (display_name, ameriflux_var, aggregation, forcing_col, unit)
var_map = [
    ("SWdown", "downward_shortwave", "mean", "dswr",       "W/m2"),
    ("LWdown", "downward_longwave",  "mean", "dlwr",       "W/m2"),
    ("Tair",   "air_temp",           "mean", "temp_c",     "C"),
    ("Wind",   "wind_speed",         "mean", "wind_speed", "m/s"),
    ("VPD",    "vapor_pressure_deficit", "mean", "vpd_hpa", "hPa"),
]

# Fetch obs and compute metrics for each variable
results = {}
obs_data = {}
print(f"Forcing quality check: {site_id} WY{water_year}\n")

for disp, obs_var, agg, forc_col, unit in var_map:
    try:
        df = hf.get_point_data(
            dataset="ameriflux", variable=obs_var,
            temporal_resolution="hourly", aggregation=agg,
            date_start=start, date_end=end, site_ids=site_id,
        )
        if df is None or df.empty or site_id not in df.columns:
            print(f"  {disp:<8}: no data available")
            continue
        obs = pd.Series(df[site_id].values.astype(float),
                        index=pd.to_datetime(df["date"])).replace(-9999, np.nan).dropna()
        # Shift to local time
        obs.index = obs.index + pd.Timedelta(hours=utc_offset)
        # Temperature: Ameriflux is in C, forcing temp_c is also C
        # But if obs_var is air_temp, Ameriflux may be in K — check range
        if obs_var == "air_temp" and obs.median() > 200:
            obs = obs - 273.15  # K -> C

        obs_data[disp] = obs
        forc = forcing_df[forc_col]

        common = forc.index.intersection(obs.index)
        f_c = forc.loc[common].dropna()
        o_c = obs.loc[common].dropna()
        common = f_c.index.intersection(o_c.index)
        if len(common) < 100:
            print(f"  {disp:<8}: insufficient overlap ({len(common)} hours)")
            continue
        f_c, o_c = f_c.loc[common], o_c.loc[common]

        bias = float((f_c - o_c).mean())
        rmse = float(np.sqrt(((f_c - o_c)**2).mean()))
        r = float(np.corrcoef(f_c, o_c)[0, 1])
        results[disp] = {"bias": bias, "rmse": rmse, "r": r, "unit": unit, "n": len(common)}
        print(f"  {disp:<8}: bias={bias:+.2f} {unit}, RMSE={rmse:.2f}, r={r:.2f} ({len(common)} hrs)")
    except Exception as e:
        print(f"  {disp:<8}: skipped ({e})")

if not results:
    print("\nNo forcing comparison data available for this site.")
else:
    # Multi-panel plot: daily time series, diurnal (May-Sep), scatter
    vars_ok = [v for v in ["SWdown", "LWdown", "Tair", "Wind", "VPD"] if v in results]
    n_vars = len(vars_ok)

    fig, axes = plt.subplots(n_vars, 3, figsize=(18, 4 * n_vars),
                             gridspec_kw={"width_ratios": [3, 1.5, 1.5]})
    if n_vars == 1:
        axes = axes.reshape(1, -1)

    forc_col_map = {"SWdown": "dswr", "LWdown": "dlwr", "Tair": "temp_c",
                    "Wind": "wind_speed", "VPD": "vpd_hpa"}

    for row, var in enumerate(vars_ok):
        m = results[var]
        forc = forcing_df[forc_col_map[var]]
        obs = obs_data[var]

        common = forc.index.intersection(obs.dropna().index)
        f_c = forc.loc[common]
        o_c = obs.loc[common].dropna()
        common = f_c.index.intersection(o_c.index)
        f_c, o_c = f_c.loc[common], o_c.loc[common]

        # Panel 1: Daily time series
        ax = axes[row, 0]
        ax.plot(o_c.resample("D").mean(), "k-", lw=1.5, label="Ameriflux")
        ax.plot(f_c.resample("D").mean(), "r-", lw=1, alpha=0.8, label="CW3E")
        ax.set_ylabel(f"{var} ({m['unit']})")
        ax.set_title(f"{var}: bias={m['bias']:+.2f}, RMSE={m['rmse']:.2f}, r={m['r']:.2f}")
        ax.legend(fontsize=8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
        ax.grid(True, alpha=0.3)

        # Panel 2: Diurnal cycle (May-Sep)
        ax = axes[row, 1]
        gs = (common.month >= 5) & (common.month <= 9)
        if gs.sum() > 100:
            ax.plot(o_c.loc[common[gs]].groupby(o_c.loc[common[gs]].index.hour).mean(),
                    "k-", lw=2, label="Obs")
            ax.plot(f_c.loc[common[gs]].groupby(f_c.loc[common[gs]].index.hour).mean(),
                    "r-", lw=1.5, alpha=0.8, label="CW3E")
            ax.set_xlabel("Local Hour")
            ax.set_title("Diurnal (May-Sep)")
            ax.set_xticks(range(0, 24, 6))
            ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        # Panel 3: Scatter
        ax = axes[row, 2]
        step = max(1, len(f_c) // 2000)
        ax.scatter(o_c.values[::step], f_c.values[::step], s=3, alpha=0.3, color="steelblue")
        lims = [min(o_c.min(), f_c.min()), max(o_c.max(), f_c.max())]
        ax.plot(lims, lims, "k--", alpha=0.5)
        ax.set_xlabel(f"Obs ({m['unit']})")
        ax.set_ylabel(f"CW3E ({m['unit']})")
        ax.set_title(f"r={m['r']:.2f}")
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.3)

    fig.suptitle(f"{site_id} WY{water_year} — CW3E Forcing vs Ameriflux Observations",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

clm_ds.close()